# KrickBot Llama 3.1 8B Fine-Tuning (Google Colab)

Fine-tunes Llama 3.1 8B Instruct in 4-bit with `unsloth` + QLoRA, on a free Colab T4.

### How to use
1. **Runtime > Change runtime type > T4 GPU.**
2. Put `refined_dataset.jsonl` in your Google Drive at `MyDrive/krickbot/`.
3. Run all cells top to bottom.
4. Download the exported `.gguf` file from Drive once export finishes.

### Checkpoint & resume (avoids wasted Colab time)
- Checkpoints save to **Google Drive** every `SAVE_STEPS` steps, so they survive a Colab
  disconnect/timeout even though local disk is wiped between sessions.
- If training stops (crash, 90-min idle timeout, 12hr session cap), just **re-run all cells**.
  The training cell auto-detects the latest checkpoint in Drive and resumes from it —
  it does **not** start over.
- To start completely fresh, delete `MyDrive/krickbot/outputs/` before running.
- LoRA adapters are also saved separately right after training finishes, **before** the
  GGUF export step — GGUF conversion is the most failure-prone step (relies on building
  llama.cpp inside Colab), so this way a failed export never costs you the trained weights.

### What changed vs the first draft
- Fixed install pinning (unsloth's own recommended `colab-new` extras), removed the
  unconditional HF cache wipe that forced a full model re-download on every re-run.
- Added `report_to="none"` so the run never hangs waiting for a W&B login prompt.
- Added a real train/eval split + `eval_strategy` + early stopping, since the dataset is
  fairly template-repetitive and can overfit past a couple epochs.
- Added `group_by_length` (and optional `packing`) since most examples are much shorter
  than `max_seq_length=2048`, so padding was wasting a lot of compute.
- Added a safety-net adapter save + a quick inference sanity check before GGUF export.


In [ ]:
# 0. Configuration — change things here, not scattered through the notebook
BASE_MODEL   = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # matches the finalized project decision (Llama 3.1 8B QLoRA)
CHAT_TEMPLATE = "llama-3.1"

DRIVE_ROOT   = "/content/drive/MyDrive/krickbot"
DATASET_PATH = f"{DRIVE_ROOT}/refined_dataset.jsonl"
OUTPUT_DIR   = f"{DRIVE_ROOT}/outputs"          # trainer checkpoints (resumable)
ADAPTER_DIR  = f"{DRIVE_ROOT}/lora_model"        # safety-net save, right after training
GGUF_DIR     = f"{DRIVE_ROOT}/krickbot_model"    # final export

MAX_SEQ_LENGTH = 2048
LORA_RANK      = 16
LORA_ALPHA     = 16

NUM_EPOCHS       = 3           # eval + early stopping will cut this short if it stalls
SAVE_STEPS       = 250
EVAL_STEPS       = 250
EARLY_STOP_PATIENCE = 3        # stop if eval_loss doesn't improve for this many evals
LEARNING_RATE     = 2e-4
PER_DEVICE_BATCH  = 4          # bump-able from 2 -> 4 on T4 since examples are short
GRAD_ACCUM_STEPS  = 4          # effective batch size = PER_DEVICE_BATCH * GRAD_ACCUM_STEPS
USE_PACKING       = True       # packs short examples together to cut padding waste; set False if you hit issues

# GGUF export quantization. q8_0 = higher quality/larger, q4_k_m = smaller/faster for constrained deployment.
GGUF_QUANT_METHODS = ["q8_0", "q4_k_m"]


In [ ]:
# 1. Install dependencies
# Pinned to unsloth's own recommended Colab install pattern — this combination is the
# one most likely to actually resolve cleanly on Colab's current CUDA/torch build.
# If this cell fails, check https://github.com/unslothai/unsloth for the current pins
# before touching anything else — dependency breakage here is the #1 cause of "it worked
# last month and now it doesn't" on Colab notebooks like this.

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes xformers
!pip install -q sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer


In [ ]:
# 2. Environment setup
import os

# Faster HF downloads
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Prevent SFTTrainer from ever trying to prompt for a W&B API key and hanging headless
os.environ["WANDB_DISABLED"] = "true"

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.utils.import_utils").setLevel(logging.ERROR)

# NOTE: intentionally NOT doing `rm -rf ~/.cache/huggingface/hub` here.
# That forces a full multi-GB re-download of the base model on every re-run of this
# cell within a session. If you ever hit a corrupted-download error, run this manually
# in a scratch cell instead of baking it into every run:
#   !rm -rf ~/.cache/huggingface/hub


In [ ]:
# 3. Mount Google Drive & verify dataset
from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(DATASET_PATH), f"Dataset not found at {DATASET_PATH}"
print(f"\u2705 Dataset found: {DATASET_PATH}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)


In [ ]:
# 4. Load model & tokenizer, attach LoRA adapters
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,          # auto-detect (bf16 on T4-class+ hardware that supports it)
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


In [ ]:
# 5. Prepare dataset — apply chat template, then hold out an eval split
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = CHAT_TEMPLATE,
    # Your dataset already uses "role"/"content" keys with values "system"/"user"/"model" —
    # this tells unsloth's formatter to treat role=="model" as the assistant turn.
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "model"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

print("Loading refined_dataset.jsonl from Google Drive...")
raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
raw_dataset = raw_dataset.map(formatting_prompts_func, batched=True)
print(f"\u2705 Dataset loaded: {len(raw_dataset)} examples")

# Held-out eval split — needed to actually see overfitting rather than flying blind on
# training loss alone. Given the dataset leans template-heavy, this matters more than usual.
split = raw_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Quick sanity check on one formatted example
print("\n--- Example formatted training text ---")
print(train_dataset[0]["text"][:800])


In [ ]:
# 6. Train (with checkpointing & auto-resume from Google Drive)
import glob
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# --- Checkpoint resume detection ---
resume_from = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = sorted(
        glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1])
    )
    if checkpoints:
        resume_from = checkpoints[-1]
        print(f"\n\U0001f504 Resuming training from: {resume_from}")
        print(f"   (Found {len(checkpoints)} checkpoint(s) in Drive)\n")
    else:
        print("\n\U0001f195 No checkpoints found. Starting fresh training.\n")
else:
    print("\n\U0001f195 No output directory found. Starting fresh training.\n")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = PER_DEVICE_BATCH,
        per_device_eval_batch_size = PER_DEVICE_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        warmup_steps = 5,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        report_to = "none",              # never hang waiting on a W&B prompt

        # --- Efficiency: most examples are much shorter than max_seq_length ---
        group_by_length = True,          # batches similar-length sequences -> less padding waste
        packing = USE_PACKING,           # concatenates short examples to fill the context window

        # --- Checkpoint settings (Drive-backed, survives disconnects) ---
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 3,

        # --- Eval, so you can see overfitting instead of flying blind ---
        eval_strategy = "steps",
        eval_steps = EVAL_STEPS,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        # --- SFT-specific settings ---
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
    ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

print("\n\u2705 Training complete!")
print(f"   Total steps: {trainer_stats.global_step}")
print(f"   Final training loss: {trainer_stats.training_loss:.4f}")


In [ ]:
# 7. Safety-net save — do this BEFORE attempting GGUF export.
# GGUF conversion depends on building llama.cpp inside Colab and breaks across unsloth
# versions more often than the training step does. Save adapters first so a failed
# export never costs you the actual trained weights.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\u2705 LoRA adapters safely saved to {ADAPTER_DIR}")


In [ ]:
# 8. Quick inference sanity check before spending time on GGUF export
FastLanguageModel.for_inference(model)

test_prompt = [
    {"role": "system", "content": "You are KrickBot, an analytical cricket assistant. Answer the user's question using ONLY the provided facts. Write naturally and avoid robotic repetition."},
    {"role": "user", "content": "What is Babar Azam's recent batting form?\n\n[FACTS: Player: Babar Azam | Last N Matches: 5 | Runs: 210 | Avg: 42.00 | SR: 88.50]"},
]

inputs = tokenizer.apply_chat_template(test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(input_ids=inputs, max_new_tokens=120, use_cache=True, temperature=0.7)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

# Sanity checks worth eyeballing here:
#  - Did it actually use the numbers from FACTS (210, 42.00, 88.50) rather than inventing new ones?
#  - Does it stay in KrickBot's voice rather than drifting into generic chatbot phrasing?
# If either looks off, it's worth catching now rather than after a slow GGUF export.


In [ ]:
# 9. Export to GGUF for local deployment (llama.cpp / Ollama)
# This step is the most likely to fail due to llama.cpp build issues inside Colab.
# Adapters are already safely saved in step 7, so a failure here is just a retry, not a loss.
for quant in GGUF_QUANT_METHODS:
    try:
        model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=quant)
        print(f"\u2705 Exported GGUF ({quant}) to {GGUF_DIR}")
    except Exception as e:
        print(f"\u26a0\ufe0f GGUF export failed for {quant}: {e}")
        print("   Your LoRA adapters are still safe in", ADAPTER_DIR)
        print("   You can retry export later by reloading the adapters instead of retraining.")
